<a href="https://colab.research.google.com/github/pranuk050-pixel/Pyspark_Programming/blob/main/27_08_26_Spark_Rdd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("sample_RDD").getOrCreate()
number =[10,20,30,40,50]

sc = spark.sparkContext

rdd = sc.parallelize(number)
print(type(rdd))
print(rdd.collect())

#  mulitply each value in rdd with 5

rdd2= rdd.map(lambda x: x*5)
print(rdd2.collect())


rdd2= rdd.map(lambda i: i +5)
print(rdd2.collect())

# create new RDD with values >= 40
rdd3 = rdd.filter( lambda x : x >= 40)
print(rdd3.collect())

strings = ['hello', 'students', 'pyspark']

str_rdd= sc.parallelize(strings)
upp_rdd = str_rdd.map(lambda x : len(x))
print(upp_rdd.collect())
print(upp_rdd.count())

l1 =['hi students, who are you? pyspark session started']

rdd1= sc.parallelize(l1)
rdd2= rdd1.flatMap(lambda x: x.split(' '))
print(rdd2.collect())
print(rdd2.count())

s1='hi students, who are you? pyspark session started'
print(s1.split(' '))

l1 =[['hi students,'], ['who are you?'], ['pyspark session'], ['started']]

rdd1= sc.parallelize(l1)
rdd2= rdd1.flatMap(lambda x: x[0].split(' '))
print(rdd2.collect())
print(rdd2.count())
print(rdd2.take(6))
print(rdd2.first())

l1 =[['hi students, hi'], ['hi hello you?'], ['pyspark session'], ['pyspark']]

rdd1= sc.parallelize(l1)
# print how many times each word is presented on RDD
rdd2= rdd1.flatMap(lambda x: x[0].split(' '))


map_rdd = rdd2.map( lambda x : (x,1))
red_rdd = map_rdd.reduceByKey( lambda a,b : a+b)
print(red_rdd.collect())

grp_rdd = rdd2.groupBy(lambda word:word)
res_rdd = grp_rdd.mapValues(len)
print(res_rdd.collect())

l=[10,20,30,405,60,708,8]

num_rdd = sc.parallelize(l)
print(num_rdd.collect())

res_rdd = num_rdd.reduce(lambda a,b : a+b)
print(res_rdd)

<class 'pyspark.core.rdd.RDD'>
[10, 20, 30, 40, 50]
[50, 100, 150, 200, 250]
[15, 25, 35, 45, 55]
[40, 50]
[5, 8, 7]
3
['hi', 'students,', 'who', 'are', 'you?', 'pyspark', 'session', 'started']
8
['hi', 'students,', 'who', 'are', 'you?', 'pyspark', 'session', 'started']
['hi', 'students,', 'who', 'are', 'you?', 'pyspark', 'session', 'started']
8
['hi', 'students,', 'who', 'are', 'you?', 'pyspark']
hi
[('students,', 1), ('hello', 1), ('session', 1), ('hi', 3), ('you?', 1), ('pyspark', 2)]
[('students,', 1), ('hello', 1), ('session', 1), ('hi', 3), ('you?', 1), ('pyspark', 2)]
[10, 20, 30, 405, 60, 708, 8]
1241


1.differences between MAP and FLATMAP  

Differences in Paragraph Pointers
map() → Applies a function to each element of the RDD and returns a new RDD where each input element corresponds to exactly one output element.

flatMap() → Applies a function that can return multiple values (like a list or iterator) for each input element, and then flattens them into a single RDD.

Transformation Type → map() is one-to-one, while flatMap() is one-to-many.

Output Structure → map() keeps the structure intact (no flattening), whereas flatMap() merges nested results into a single sequence.

Use Case → Use map() for simple element-wise transformations (e.g., square each number). Use flatMap() when splitting or expanding elements (e.g., splitting sentences into words).

Resulting RDD → map() → RDD of transformed elements; flatMap() → RDD of flattened elements.

In [3]:
from pyspark import SparkContext
#sc = SparkContext("local", "Map vs FlatMap")

# Sample RDD
rdd = sc.parallelize(["hello world", "spark rdd"])

# Using map()
mapped = rdd.map(lambda x: x.split(" "))
print(mapped.collect())
# Output: [['hello', 'world'], ['spark', 'rdd']]

# Using flatMap()
flatmapped = rdd.flatMap(lambda x: x.split(" "))
print(flatmapped.collect())
# Output: ['hello', 'world', 'spark', 'rdd']


[['hello', 'world'], ['spark', 'rdd']]
['hello', 'world', 'spark', 'rdd']


Use map() when each input produces exactly one output.

Use flatMap() when each input can produce multiple outputs and you want them flattened into a single RDD.

2.How does the collect function works in Spark Rdd

In PySpark RDDs, the collect() function is used to retrieve all elements of the RDD from the distributed cluster back to the driver program as a Python list. It’s a powerful but potentially dangerous operation if the dataset is very large, because it loads everything into the driver’s memory.

Purpose → Brings the entire RDD data from worker nodes to the driver program.

Return Type → Returns a standard Python list containing all elements of the RDD.

Usage → Commonly used for debugging, testing, or when the dataset is small enough to fit in driver memory.

Performance Risk → Collecting large RDDs can cause driver memory overflow and crash the application.

Alternative → Use take(n) or top(n) when you only need a subset of data instead of the entire dataset.

Operation → Executes the DAG (Directed Acyclic Graph) of transformations and triggers computation, since it’s an action in Spark.

In PySpark RDDs, both reduceByKey and groupByKey are used for operations on pair RDDs (key-value pairs), but they differ in efficiency and behavior. Let’s break it down in paragraph pointers and a comparison table.

reduceByKey → Combines values for each key using a specified function (like sum, max, etc.) before shuffling data across the cluster. This reduces network traffic and is more efficient.

groupByKey → Groups all values associated with a key into an iterable, then sends them across the network. This can cause heavy data shuffling and memory overhead.

Efficiency → reduceByKey is aggregation-friendly and optimized, while groupByKey is slower and should be avoided for large datasets.

Use Case → Use reduceByKey when you want to aggregate values (e.g., sum of sales per store). Use groupByKey when you need to collect all values for a key without aggregation (e.g., list of transactions per customer).

Operation → reduceByKey applies the function locally on each partition first, then combines results globally. groupByKey simply groups values without reducing them.

In [4]:
from pyspark import SparkContext
#sc = SparkContext("local", "ReduceByKey vs GroupByKey")

# Sample RDD
rdd = sc.parallelize([("a", 1), ("b", 2), ("a", 3), ("b", 4)])

# Using reduceByKey (aggregation)
reduced = rdd.reduceByKey(lambda x, y: x + y)
print(reduced.collect())
# Output: [('a', 4), ('b', 6)]

# Using groupByKey (grouping)
grouped = rdd.groupByKey()
print([(k, list(v)) for k, v in grouped.collect()])
# Output: [('a', [1, 3]), ('b', [2, 4])]


[('b', 6), ('a', 4)]
[('b', [2, 4]), ('a', [1, 3])]


Use reduceByKey for aggregation (preferred, efficient).

Use groupByKey only when you need all values grouped without aggregation.

📌 Common RDD Inbuilt Functions (Actions)
collect() → Returns all elements to driver.

count() → Returns number of elements.

take(n) → Returns first n elements.

reduce() → Aggregates all elements using a function.

saveAsTextFile() → Saves RDD to storage.

In [5]:
# map() → Applies a function to each element.
rdd=sc.parallelize([1,2,3])
rdd.map(lambda x: x*2).collect()


[2, 4, 6]

In [11]:
# flatMap() → Applies a function that returns multiple values and flattens them.
rdd=sc.parallelize(["Hello World"])
rdd.flatMap(lambda x:x.split(" ")).collect()

['Hello', 'World']

In [13]:
# filter() → Keeps elements that satisfy a condition.

rdd=sc.parallelize([1,2,3,4,5])
rdd.filter(lambda x: x%2 ==0).collect()


[2, 4]

In [15]:
# reduceByKey() → Aggregates values per key using a function.
rdd=sc.parallelize([('a',1),('b',2),('a',2),('c',1),('b',3)])
rdd.reduceByKey(lambda x ,y : x+y).collect()

[('b', 5), ('c', 1), ('a', 3)]

In [ ]:
# groupByKey() → Groups values per key into an iterable.
rdd = sc.parallelize([("a",1),("a",2),("b",3)])
[(k,list(v)) for k,v in rdd.groupByKey().collect()]
# [('a',[1,2]),('b',[3])]


In [16]:
# distinct() → Removes duplicates.
rdd = sc.parallelize([1,2,2,3])
rdd.distinct().collect()  # [1,2,3]


[2, 1, 3]

📌 Common RDD Inbuilt Functions (Actions)
collect() → Returns all elements to driver.

count() → Returns number of elements.

take(n) → Returns first n elements.

reduce() → Aggregates all elements using a function.

saveAsTextFile() → Saves RDD to storage.